In [ ]:
import glob
import os
import pandas as pd
import re
import sys

if '../' not in sys.path:
    sys.path.append('../')

from emp.dataset import extract_bl_shelfmark, gen_prompt

In [ ]:
def extract_title(s):
    date_re = re.compile(r"\d{4,4}")
    if date_re.search(s.split()[-1]) and s.split()[-2] in ["a", "b", "c"]:
        return " ".join(s.split()[:-2])
    elif date_re.search(s.split()[-1]):
        return " ".join(s.split()[:-1]) 
    elif s.split()[-1] in ["a", "b", "c"]:
        return " ".join(s.split()[:-1])
    else:
        return s

In [ ]:
def extract_edition(s):
    date_re = re.compile(r"\d{4,4}")
    if date_re.search(s.split()[-1]) and s.split()[-2] in ["a", "b", "c"]:
        return s.split()[-2]
    elif date_re.search(s.split()[-1]):
        return s.split()[-1] 
    elif s.split()[-1] in ["a", "b", "c"]:
        return s.split()[-1]
    else:
        return None

### Boll EMP

#### Boll EMP df

In [ ]:
# 1 Confirmed duplicate row for Miftah al-Bayan 1890
boll_emp_df = pd.read_csv("../data/external/boll_emp_by_date.csv", encoding="utf8", skipfooter=3, engine='python', usecols=[0, 1, 2, 3]).drop_duplicates(subset=["shelfmark", "title_edition"])
boll_emp_df["shelfmark"] = boll_emp_df["shelfmark"].str.strip()
boll_emp_df.set_index("shelfmark", inplace=True)
boll_emp_df.rename(index={"14625.d. 16": "14625.d.16"}, inplace=True)
assert boll_emp_df.index[-1] == "14625.e.5"
assert boll_emp_df.reset_index().value_counts(subset=["shelfmark", "title_edition"]).max() == 1

In [ ]:
# modifications to boll_emp_df
# 22/06/26 have double checked all these corrections are applied to the correct cells
boll_emp_df.loc["14620.k.1", "title_edition"] = "Anbiya 1897"
boll_emp_df.loc["14620.g.11", "title_edition"] = "Bidayat al-Salikin 1906"
boll_emp_df.loc["14620.d.20(1)", "title_edition"] = "Bustan Arifin 1820-1822"
boll_emp_df.loc["14625.d.14", "title_edition"] = "Dandan Setia 1894"
boll_emp_df.loc["14625.c.47", "title_edition"] = "Dermah Tasiah 1906"
boll_emp_df.loc["14626.c.14(4)", "title_edition"] = "Haris Fadhillah 1900"
boll_emp_df.loc["14620.ff.1", "title_edition"] = "Hidayat al-Awamm 1903"
boll_emp_df.loc["14620.g.17", "title_edition"] = "Hilyat al-Anam 1893"
boll_emp_df.loc["Jav.76", "title_edition"] = "Husn al-Akhlak 1900"
boll_emp_df.loc["14620.b.14(8)", "title_edition"] = "Ilmu Kepandaian 1843"
boll_emp_df.loc["14629.e.2", "title_edition"] = "Ilmu Kepandaian a 1839"
boll_emp_df.loc["14519.d.44(1)", "title_edition"] = "Maulud 1871.a"
boll_emp_df.loc["14519.d.9(3)", "title_edition"] = "Maulud 1871.b"
boll_emp_df.loc["14620.g.7", "title_edition"] = "Miftah al-Bayan 1890"
boll_emp_df.loc["14620.d.17(4)", "title_edition"] = "Pelajaran Bahasa Melayu (No.1) b 1838/9"
boll_emp_df.loc["14620.f.5", "title_edition"] = "Tahsil al-Ujur 1893"
boll_emp_df.loc["14620.g.5", "title_edition"] = "Targhib al-Nas 1873"
boll_emp_df.loc["14626.a.37", "title_edition"] = "Umm al-Burhan a"

In [ ]:
boll_emp_df["title"] = boll_emp_df["title_edition"].apply(lambda x: extract_title(x))
boll_emp_df["edition"] = boll_emp_df["title_edition"].apply(lambda x: extract_edition(x))

#### title_loc_df

In [ ]:
title_loc_path = "../data/interim/title_loc_df.csv"
title_loc_df = pd.read_csv(title_loc_path, encoding="utf-8-sig", index_col=0)

In [ ]:
entry = "Pelajaran Bahasa Arab"
# gen_prompt(title_loc_df.loc[entry, "entry_text"], entry)

In [ ]:
title_loc_df.index

In [ ]:
assert not [x for x in boll_emp_df.set_index(["title"]).index.difference(title_loc_df.index)]
assert not boll_emp_df["edition"].hasnans
assert title_loc_df.index.is_unique

In [ ]:
title_loc_df["entry_start"].value_counts().head(8)

#### boll_entry_df

In [ ]:
boll_entry_df = title_loc_df.merge(boll_emp_df, how="inner", left_index=True, right_on="title").reset_index()
assert boll_entry_df.drop_duplicates(subset="title")["entry_start"].is_monotonic_increasing
boll_entry_df.set_index("title", inplace=True)
assert not boll_entry_df["entry_text"].hasnans

In [ ]:
boll_entry_df["extracted_shelfmark"] = boll_entry_df["entry_text"].apply(lambda x: extract_bl_shelfmark(x)).str.strip().str.replace("v. ", "v.")

In [ ]:
absent_raw_sm_df = boll_entry_df[~(boll_entry_df.apply(lambda x: x["shelfmark"] in x["entry_text"], axis=1))]
absent_sm_df = absent_raw_sm_df[absent_raw_sm_df["shelfmark"] != absent_raw_sm_df["extracted_shelfmark"]]

In [ ]:
absent_sm_df

In [ ]:
# checking that extract_bl_shelfmark will find the correct shelfmark
# correct shelfmark -> shelfmark in entry -> entry sm will be corrected by extract_bl_shelfmark

# 14620.k.1 -> 14620.k.l -> yes
# Jav.73 -> Jav. 73 -> yes

# 14653.d.1 -> 14653.d.l -> no  # but only because there are multiple shelfmarks for this edition, will probably have to pick this up in post

# Jav.70 -> Jav. 70 -> yes

# 14626.d.11(2) -> 14626.d.ll(2) -> yes

# 14620.c.10 -> 14620.c.1O -> yes

# ORB.30/612 -> ORB. 30/612 -> yes

# 14653.b.1 -> 14653.b.l -> yes

# 14620.d.19(14) -> 14620.d.19(l4) -> yes

# 14625.a.1 -> 14625.a.l -> yes

# 14626.d.11(6) -> 14626.d.ll(6) -> yes (unclear why not picked up already)

# fixed by modifying extract_bl_shelfmark

# 14626.c.10 -> 14626.c.1O -> yes
# 14626.a.10 -> 14626.a.1O -> yes
# 14625.e.10 -> 14625.e.1O -> yes
# 14626.d.11(10) -> 14626.d.11(1O) -> yes
# 14626.d.10 -> 14626.d.1O -> yes
# ORB.30/452 -> ORB. 30/452 -> yes

# fixed by modifying entry in extract_clean_entries

# 14620.a.19(10) -> 14620.aI9(10) -> no  # sm missing a '.', will need to add in
# 14620.b.18(10) -> 14620.b.18{l0) -> no  # will need to modify shelfmark
# 14623.c.4 -> 14623.cA -> no  # need to modify shelfmark
# 14626.d.11(8) -> 14626.d.l1 (8) -> no  # may need to modify sm to remove space
# 14625.a.9 -> 14625.a9 -> no  # may need to add missing '.'
# 14620.g.20(0) -> 14620.g.20(-) -> no  # may need to modify sm to swap -/0
# 14626.e.4 -> 14626.eA -> no  # will need to add '.' to sm and swap A/4
# 14633.a.38 -> 1463303.38 -> no  # will need to fix sm

# 14620.g.11 -> -> no  # may need to modify entry extent
# 14620.d.14 -> -> no  # may need to modify entry extend
# 14625.e.13 -> -> no  # need to modify entry extent
# 14629.c.31 -> -> no  # no, may need to modify entry
# 14620.g.19(9) ->  -> no  # may need to modify entry extent
# 14620.d.11 ->  -> no  # may need to modify entry

In [ ]:
# with open("../data/interim/sm_check.txt", "w") as f:
#     [f.write(r[1]["shelfmark"] + "\n" + r[1]["extracted_shelfmark"] + "\n\n" + r[1]["entry_text"] + "\n\n") for r in absent_sm_df.iterrows()]

In [ ]:
assert len(absent_sm_df) == 11  # I've accounted for these 17

In [ ]:
boll_entry_df[~(boll_entry_df.apply(lambda x: x["edition"] in x["entry_text"], axis=1))]

### Export prepared entries to interim

In [ ]:
boll_entry_df.drop_duplicates(subset="title").to_csv("../data/interim/title_dedup_boll_emp.csv", encoding="utf-8-sig", index=False)

In [ ]:
# pd.read_csv("../data/interim/title_dedup_boll_emp.csv", encoding="utf-8-sig").set_index("title")

### AAC

In [ ]:
full_aac_df = pd.read_csv("../data/external/full_aac_by_date.csv", encoding="utf8", usecols=[0,1,2,3]).drop_duplicates(subset=["shelfmark", "title_edition", "year"])
full_aac_df["shelfmark"] = full_aac_df["shelfmark"].apply(
    lambda x: x.split(" (IOLR")[0]
).str.strip(
).str.replace("° ", ""
).str.replace(". ", "."
).str.replace(".3.", ".a."
).str.replace(".33.", ".aa.")

full_aac_df.set_index("shelfmark", inplace=True)
full_aac_df.rename(index={
    "14625.d. 16": "14625.d.16",
    "14625.e,17(1)": "14625.e.17(1)",
    "l": "14626.c.16(1)",
    "14620.g.20(-)": "14620.g.20(0)",
    "14620.f.3(I)": "14620.f.3(1)",
}, inplace=True)

# Verified 30/06/26
full_aac_df.loc["14620.k.1", "title_edition"] = "Anbiya 1897"
full_aac_df.loc["14620.g.11", "title_edition"] = "Bidayat al-Salikin 1906"
full_aac_df.loc["14620.d.20(1)", "title_edition"] = "Bustan Arifin 1820-1822"
full_aac_df.loc["14625.d.14", "title_edition"] = "Dandan Setia 1894"
full_aac_df.loc["14625.c.47", "title_edition"] = "Dermah Tasiah 1906"
full_aac_df.loc["14620.ff.1", "title_edition"] = "Hidayat al-Awamm 1903"
full_aac_df.loc["14620.g.17", "title_edition"] = "Hilyat al-Anam 1893"
full_aac_df.loc["Jav.76", "title_edition"] = "Husn al-Akhlak 1900"
full_aac_df.loc["14620.b.14(8)", "title_edition"] = "Ilmu Kepandaian 1843"
full_aac_df.loc["14629.e.2", "title_edition"] = "Ilmu Kepandaian a 1839"
full_aac_df.loc["14519.d.44(1)", "title_edition"] = "Maulud 1871.a"
full_aac_df.loc["14519.d.9(3)", "title_edition"] = "Maulud 1871.b"
full_aac_df.loc["14620.g.7", "title_edition"] = "Miftah al-Bayan 1890"
full_aac_df.loc["14620.d.17(4)", "title_edition"] = "Pelajaran Bahasa Melayu (No.1) b 1838/9"
full_aac_df.loc["14625.aa.2(1)", "title_edition"] = "Sabar Ali 1851"
full_aac_df.loc["14620.f.5", "title_edition"] = "Tahsil al-Ujur 1893"
full_aac_df.loc["14620.g.5", "title_edition"] = "Targhib al-Nas 1873"
full_aac_df.loc["14626.a.37", "title_edition"] = "Umm al-Burhan a"
full_aac_df.loc["14625.b.1", "title_edition"] = "Yue Fei 1891"

assert boll_emp_df.loc[boll_emp_df.index.difference(full_aac_df.index)].empty
assert full_aac_df.reset_index().value_counts(subset=["shelfmark", "title_edition"]).max() == 1

In [ ]:
non_boll_aac_df = full_aac_df.set_index("title_edition", append=True).loc[full_aac_df.set_index("title_edition", append=True).index.difference(boll_emp_df.set_index("title_edition", append=True).index)]
assert len(non_boll_aac_df) + len(boll_emp_df) == len(full_aac_df)

In [ ]:
full_aac_df["title"] = full_aac_df["title_edition"].apply(lambda x: extract_title(x))
full_aac_df["edition"] = full_aac_df["title_edition"].apply(lambda x: extract_edition(x))

In [ ]:
full_aac_df.set_index(["title"]).loc[full_aac_df.set_index(["title"]).index.difference(title_loc_df.index)].head(50)

In [ ]:
assert not full_aac_df.set_index(["title"]).index.difference(title_loc_df.index).empty
assert not full_aac_df["edition"].hasnans
assert title_loc_df.index.is_unique

In [ ]:
full_aac_df.set_index(["title"]).index.difference(title_loc_df.index).empty